# LSEG Data Pull - AnalystBased

Dieses Notebook startet mit identischem `Setup` und identischer `Input + Run-Konfiguration` wie das NetPayout-Notebook.


## 0. Setup



In [ ]:
from pathlib import Path
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lseg_series_puller import (
    SeriesPullConfig,
    SeriesFieldSpec,
    run_standard_pull,
)
from plot_style import COLORS, set_global_plot_style, style_axes, style_legend, style_time_axis

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")
pd.set_option("display.max_columns", 200)
set_global_plot_style()


## 1. Input + Run-Konfiguration

Geladene Basisdaten:
- `Project_Data/intermediate/euro500.parquet`

Verwendete Schlüsselspalten:
- `firm_id`
- `date`
- (ID-Fallback aus vorhandenen Spalten wie `ISIN`, `RIC_current`, `RIC`, `id_type`, `pull_id`)



In [ ]:
from project_paths import BASE_DIR, DATA_DIR, CACHE_DATA_DIR

BASE_PATH = DATA_DIR / 'euro500.parquet'
OUTPUT_PATH = DATA_DIR / 'euro500_analystbased.parquet'

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Missing file: {BASE_PATH}')

base = pd.read_parquet(BASE_PATH).copy()
if 'date' not in base.columns or 'firm_id' not in base.columns:
    raise ValueError('euro500.parquet must contain at least firm_id and date columns.')

base['date'] = pd.to_datetime(base['date'], errors='coerce').dt.normalize()
base = base.dropna(subset=['firm_id']).copy()

# ---------- Candidate overrides from full ID history ----------
ALL_ID_CANDIDATE_OVERRIDES = {}

for firm_id, g in base.groupby('firm_id', sort=False):
    seen = set()
    cands = []

    def _add(it: str, v: object) -> None:
        if pd.isna(v):
            return
        val = str(v).strip()
        if not val:
            return
        typ = str(it).strip().upper()
        if typ == 'ISIN' and val.upper().startswith('ISIN:'):
            val = val.split(':', 1)[1].strip()
        if typ not in {'ISIN', 'RIC'} or not val:
            return
        key = (typ, val)
        if key in seen:
            return
        seen.add(key)
        cands.append(key)

    for col, typ in [('ISIN', 'ISIN'), ('RIC_current', 'RIC'), ('RIC', 'RIC')]:
        if col in g.columns:
            for v in g[col].dropna().astype(str):
                _add(typ, v)

    if {'id_type', 'pull_id'}.issubset(g.columns):
        hist = g[['id_type', 'pull_id']].dropna()
        for _, row in hist.iterrows():
            _add(str(row['id_type']).upper().strip(), row['pull_id'])

    if cands:
        ALL_ID_CANDIDATE_OVERRIDES[str(firm_id)] = cands

# ---------- Universal Pull Panel (all firms, fixed range) ----------
PULL_START_DATE = pd.Timestamp('1995-12-31')
PULL_END_DATE = pd.Timestamp('2025-12-31')
PULL_FREQ = 'YE-DEC'


def _first_valid(series: pd.Series):
    x = series.dropna()
    return x.iloc[0] if not x.empty else pd.NA


id_cols = [c for c in ['ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id', 'company_name'] if c in base.columns]
firm_meta = (
    base[['firm_id', *id_cols]].copy()
    .groupby('firm_id', as_index=False)
    .agg({c: _first_valid for c in id_cols})
)

pull_dates = pd.date_range(PULL_START_DATE, PULL_END_DATE, freq=PULL_FREQ).normalize()
pull_calendar = pd.DataFrame({'date': pull_dates})
firm_meta['_k'] = 1
pull_calendar['_k'] = 1
pull_base = firm_meta.merge(pull_calendar, on='_k', how='inner').drop(columns=['_k'])
pull_base = pull_base[['firm_id', 'date', *id_cols]].sort_values(['firm_id', 'date']).reset_index(drop=True)

cand_counts = pd.Series({k: len(v) for k, v in ALL_ID_CANDIDATE_OVERRIDES.items()})
print('Loaded source index:', BASE_PATH)
print('source rows:', len(base), '| companies:', base['firm_id'].nunique())
print('source date range:', base['date'].min(), '->', base['date'].max())
print('Universal pull panel rows:', len(pull_base), '| companies:', pull_base['firm_id'].nunique())
print('pull date range:', pull_base['date'].min(), '->', pull_base['date'].max(), '| freq:', PULL_FREQ)
if not cand_counts.empty:
    print('ID candidate overrides: firms=', len(cand_counts), '| median=', int(cand_counts.median()), '| p95=', int(cand_counts.quantile(0.95)), '| max=', int(cand_counts.max()))

# ---------- Run control ----------
RESET_PULL_STATE = False
FORCE_REFRESH_ALL = False
RUN_LSEG_PULL = True
BATCH_SIZE = 10
ASOF_TOL_DAYS = 365
DEBUG_RAW_FIRST_N = 0
PRE_INDEX_PREFETCH_YEARS = 0

# Wenn True, wird auch das finale Output-Parquet entfernt und neu aufgebaut.
RESET_OUTPUT_FILE = False


## 2. Reset (optional, für kompletten Neu-Pull)

Wenn aktiviert, werden Cache/Checkpoint/Bad-ID Logs für den Analyst-Pull entfernt.


In [ ]:
ANALYST_CORE_MODULE = {
    'cache_dir': CACHE_DATA_DIR / 'analyst_core_cache_by_company_id',
    'step_rows_path': CACHE_DATA_DIR / 'analyst_core_step_rows.parquet',
    'checkpoint_path': CACHE_DATA_DIR / 'analyst_core_checkpoint.json',
    'bad_ids_path': CACHE_DATA_DIR / 'analyst_core_bad_ids.csv',
    'bad_rows_log_path': CACHE_DATA_DIR / 'analyst_core_bad_rows.parquet',
}

ANALYST_LTG_MODULE = {
    'cache_dir': CACHE_DATA_DIR / 'analyst_ltg_cache_by_company_id',
    'step_rows_path': CACHE_DATA_DIR / 'analyst_ltg_step_rows.parquet',
    'checkpoint_path': CACHE_DATA_DIR / 'analyst_ltg_checkpoint.json',
    'bad_ids_path': CACHE_DATA_DIR / 'analyst_ltg_bad_ids.csv',
    'bad_rows_log_path': CACHE_DATA_DIR / 'analyst_ltg_bad_rows.parquet',
}

if RESET_PULL_STATE:
    print('Resetting Analyst pull state (Core + LTG)...')
    for name, module in [('Core', ANALYST_CORE_MODULE), ('LTG', ANALYST_LTG_MODULE)]:
        if module['cache_dir'].exists():
            shutil.rmtree(module['cache_dir'])
            print(f"  [{name}] removed cache dir: {module['cache_dir']}")
        for k in ['step_rows_path', 'checkpoint_path', 'bad_ids_path', 'bad_rows_log_path']:
            fp = module[k]
            if fp.exists():
                fp.unlink()
                print(f"  [{name}] removed {k}: {fp}")

if RESET_OUTPUT_FILE and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print(f'Removed output file: {OUTPUT_PATH}')


## 3. Standard Puller Setup (Core + LTG separat)

Ziel: `TR.DPSMean`, `TR.CFPSMean` und `TR.NumberOfAnalysts` für `FY+1` bis `FY+5` im Core-Pull; `Long-Term Growth` separat als LTG-Pull (jährlich zum 31.12.).


In [ ]:
def run_analyst_module(
    source_df: pd.DataFrame,
    module: dict,
    specs: tuple[SeriesFieldSpec, ...],
    primary_output_col: str,
    *,
    diag_prefix: str,
    module_label: str,
    skip_known_bad_ids: bool = True,
    max_retries: int = 5,
) -> dict:
    cfg = SeriesPullConfig(
        batch_size=BATCH_SIZE,
        asof_tolerance_days=ASOF_TOL_DAYS,
        prefetch_start_days=int(PRE_INDEX_PREFETCH_YEARS * 365),
        debug_raw_first_n=DEBUG_RAW_FIRST_N,
        force_refresh=FORCE_REFRESH_ALL,
        cache_only=(not RUN_LSEG_PULL),
        skip_known_bad_ids=skip_known_bad_ids,
        max_retries=max_retries,
        base_sleep_sec=0.7,
        series_specs=specs,
        primary_output_col=primary_output_col,
        candidate_overrides=ALL_ID_CANDIDATE_OVERRIDES,
    )

    print('\n' + '=' * 90)
    print(f'RUN {module_label}')
    print('=' * 90)

    res = run_standard_pull(
        pull_type='series',
        source_df=source_df,
        config=cfg,
        cache_dir=module['cache_dir'],
        step_rows_path=module['step_rows_path'],
        checkpoint_path=module['checkpoint_path'],
        bad_ids_path=module['bad_ids_path'],
        bad_rows_log_path=module['bad_rows_log_path'],
        skip_filled_primary=False,
        merge_back=True,
        diag_prefix=diag_prefix,
    )

    print(f'{module_label} stats:', res['stats'])
    return res

ANALYST_CORE_SPECS = (
    SeriesFieldSpec(output_col='DPS_fy1_est',    fields=('TR.DPSMean(Period=FY1)',),          intervals=('monthly',)),
    SeriesFieldSpec(output_col='DPS_fy2_est',    fields=('TR.DPSMean(Period=FY2)',),          intervals=('monthly',)),
    SeriesFieldSpec(output_col='DPS_fy3_est',    fields=('TR.DPSMean(Period=FY3)',),          intervals=('monthly',)),
    SeriesFieldSpec(output_col='DPS_fy4_est',    fields=('TR.DPSMean(Period=FY4)',),          intervals=('monthly',)),
    SeriesFieldSpec(output_col='DPS_fy5_est',    fields=('TR.DPSMean(Period=FY5)',),          intervals=('monthly',)),
    SeriesFieldSpec(output_col='CFPS_fy1_est',   fields=('TR.CFPSMean(Period=FY1)',),         intervals=('monthly',)),
    SeriesFieldSpec(output_col='CFPS_fy2_est',   fields=('TR.CFPSMean(Period=FY2)',),         intervals=('monthly',)),
    SeriesFieldSpec(output_col='CFPS_fy3_est',   fields=('TR.CFPSMean(Period=FY3)',),         intervals=('monthly',)),
    SeriesFieldSpec(output_col='CFPS_fy4_est',   fields=('TR.CFPSMean(Period=FY4)',),         intervals=('monthly',)),
    SeriesFieldSpec(output_col='CFPS_fy5_est',   fields=('TR.CFPSMean(Period=FY5)',),         intervals=('monthly',)),
    SeriesFieldSpec(output_col='NumAnalysts_fy1', fields=('TR.NumberOfAnalysts(Period=FY1)',), intervals=('monthly',)),
    SeriesFieldSpec(output_col='NumAnalysts_fy2', fields=('TR.NumberOfAnalysts(Period=FY2)',), intervals=('monthly',)),
    SeriesFieldSpec(output_col='NumAnalysts_fy3', fields=('TR.NumberOfAnalysts(Period=FY3)',), intervals=('monthly',)),
    SeriesFieldSpec(output_col='NumAnalysts_fy4', fields=('TR.NumberOfAnalysts(Period=FY4)',), intervals=('monthly',)),
    SeriesFieldSpec(output_col='NumAnalysts_fy5', fields=('TR.NumberOfAnalysts(Period=FY5)',), intervals=('monthly',)),
)

ANALYST_LTG_SPECS = (
    SeriesFieldSpec(output_col='LTG_est', fields=('TR.LTGMean', 'TR.LTGMean(Period=FY1)'), intervals=('monthly',)),
)


## 4. Pull ausführen (Core + LTG)


In [ ]:
analyst_core = run_analyst_module(
    pull_base,
    ANALYST_CORE_MODULE,
    ANALYST_CORE_SPECS,
    primary_output_col='DPS_fy1_est',
    diag_prefix='analyst_core_',
    module_label='Analyst Core Pull',
)

core_df = analyst_core['merged_df'].copy()
core_df['date'] = pd.to_datetime(core_df['date'], errors='coerce').dt.normalize()
core_df = core_df.dropna(subset=['firm_id', 'date']).sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')

analyst_ltg = run_analyst_module(
    pull_base,
    ANALYST_LTG_MODULE,
    ANALYST_LTG_SPECS,
    primary_output_col='LTG_est',
    diag_prefix='analyst_ltg_',
    module_label='Analyst LTG Pull',
)

ltg_df = analyst_ltg['merged_df'].copy()
ltg_df['date'] = pd.to_datetime(ltg_df['date'], errors='coerce').dt.normalize()
ltg_df = ltg_df.dropna(subset=['firm_id', 'date']).sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')

analyst_df = core_df.merge(ltg_df[['firm_id', 'date', 'LTG_est']], on=['firm_id', 'date'], how='left')

# Sicherheitscheck: nur 31.12.-Stichtage behalten
analyst_df = analyst_df[(analyst_df['date'].dt.month == 12) & (analyst_df['date'].dt.day == 31)].copy()

print('Analyst panel prepared (Core + LTG): rows=', len(analyst_df), '| companies=', analyst_df['firm_id'].nunique())
preview_cols = [
    'firm_id', 'date',
    'DPS_fy1_est', 'DPS_fy2_est', 'DPS_fy3_est', 'DPS_fy4_est', 'DPS_fy5_est',
    'CFPS_fy1_est', 'CFPS_fy2_est', 'CFPS_fy3_est', 'CFPS_fy4_est', 'CFPS_fy5_est',
    'NumAnalysts_fy1', 'NumAnalysts_fy2', 'NumAnalysts_fy3', 'NumAnalysts_fy4', 'NumAnalysts_fy5',
    'LTG_est'
]
display(analyst_df[[c for c in preview_cols if c in analyst_df.columns]].head())


## 5. Shares Outstanding

Fehlende Werte in `shares_outstanding` werden wie im Implied-Notebook imputiert mit:

$$
\text{shares\_outstanding} = \frac{\text{mcap\_eur}}{\text{PriceClose}}
$$


In [ ]:
# Shares Outstanding wie im Implied-Notebook: aus base anreichern und mit mcap_eur / PriceClose imputieren
required_cols = ['shares_outstanding', 'mcap_eur', 'PriceClose']
if any(c not in analyst_df.columns for c in required_cols):
    base_cols = ['firm_id', 'date'] + [c for c in required_cols if c in base.columns]
    if len(base_cols) > 2:
        base_aux = base[base_cols].copy()
        base_aux['date'] = pd.to_datetime(base_aux['date'], errors='coerce').dt.normalize()
        base_aux = base_aux.dropna(subset=['firm_id', 'date']).sort_values(['firm_id', 'date'])
        base_aux = base_aux.drop_duplicates(['firm_id', 'date'], keep='last')
        analyst_df = analyst_df.merge(base_aux, on=['firm_id', 'date'], how='left', suffixes=('', '_base'))
        for c in required_cols:
            cb = f'{c}_base'
            if cb in analyst_df.columns:
                if c not in analyst_df.columns:
                    analyst_df[c] = analyst_df[cb]
                else:
                    analyst_df[c] = analyst_df[c].where(analyst_df[c].notna(), analyst_df[cb])
                analyst_df = analyst_df.drop(columns=[cb])

missing_required = [c for c in required_cols if c not in analyst_df.columns]
if missing_required:
    print(f'Shares Outstanding step skipped (missing columns): {missing_required}')
else:
    for c in required_cols:
        analyst_df[c] = pd.to_numeric(analyst_df[c], errors='coerce')

    n_total = len(analyst_df)
    missing_before = int(analyst_df['shares_outstanding'].isna().sum())
    coverage_before = 1.0 - (missing_before / n_total) if n_total else np.nan

    fill_mask = (
        analyst_df['shares_outstanding'].isna()
        & analyst_df['mcap_eur'].notna()
        & analyst_df['PriceClose'].notna()
        & (analyst_df['PriceClose'] != 0)
    )

    analyst_df.loc[fill_mask, 'shares_outstanding'] = (
        analyst_df.loc[fill_mask, 'mcap_eur'] / analyst_df.loc[fill_mask, 'PriceClose']
    )

    missing_after = int(analyst_df['shares_outstanding'].isna().sum())
    coverage_after = 1.0 - (missing_after / n_total) if n_total else np.nan
    filled_rows = int(fill_mask.sum())

    print('Shares outstanding coverage update:')
    print(f'- total rows: {n_total}')
    print(f'- missing before: {missing_before} ({(1-coverage_before)*100:.2f}%)')
    print(f'- missing after : {missing_after} ({(1-coverage_after)*100:.2f}%)')
    print(f'- filled rows   : {filled_rows}')


## 6. Coverage Quick Check


In [ ]:
dps_cols = ['DPS_fy1_est', 'DPS_fy2_est', 'DPS_fy3_est', 'DPS_fy4_est', 'DPS_fy5_est']
cfps_cols = ['CFPS_fy1_est', 'CFPS_fy2_est', 'CFPS_fy3_est', 'CFPS_fy4_est', 'CFPS_fy5_est']
n_cols = ['NumAnalysts_fy1', 'NumAnalysts_fy2', 'NumAnalysts_fy3', 'NumAnalysts_fy4', 'NumAnalysts_fy5']
ltg_cols = ['LTG_est']
so_cols = ['shares_outstanding']

cov_dps = analyst_df.assign(year=analyst_df['date'].dt.year).groupby('year')[dps_cols].apply(lambda g: g.notna().mean()).round(3)
cov_cfps = analyst_df.assign(year=analyst_df['date'].dt.year).groupby('year')[cfps_cols].apply(lambda g: g.notna().mean()).round(3)
cov_n = analyst_df.assign(year=analyst_df['date'].dt.year).groupby('year')[n_cols].apply(lambda g: g.notna().mean()).round(3)
cov_ltg = analyst_df.assign(year=analyst_df['date'].dt.year).groupby('year')[ltg_cols].apply(lambda g: g.notna().mean()).round(3)
cov_so = analyst_df.assign(year=analyst_df['date'].dt.year).groupby('year')[so_cols].apply(lambda g: g.notna().mean()).round(3)

print('Coverage DPSMean (tail):')
display(cov_dps.tail(15))
print('Coverage CFPSMean (tail):')
display(cov_cfps.tail(15))
print('Coverage NumberOfAnalysts (tail):')
display(cov_n.tail(15))
print('Coverage Long-Term Growth (tail):')
display(cov_ltg.tail(15))
print('Coverage Shares Outstanding (tail):')
display(cov_so.tail(15))

print('Overall non-null share DPSMean:')
display(analyst_df[dps_cols].notna().mean().round(3).to_frame('share'))
print('Overall non-null share CFPSMean:')
display(analyst_df[cfps_cols].notna().mean().round(3).to_frame('share'))
print('Overall non-null share NumberOfAnalysts:')
display(analyst_df[n_cols].notna().mean().round(3).to_frame('share'))
print('Overall non-null share Long-Term Growth:')
display(analyst_df[ltg_cols].notna().mean().round(3).to_frame('share'))
print('Overall non-null share Shares Outstanding:')
display(analyst_df[so_cols].notna().mean().round(3).to_frame('share'))


## 7. Output speichern


In [ ]:
keep_cols = [
    'firm_id', 'date',
    'DPS_fy1_est', 'DPS_fy2_est', 'DPS_fy3_est', 'DPS_fy4_est', 'DPS_fy5_est',
    'CFPS_fy1_est', 'CFPS_fy2_est', 'CFPS_fy3_est', 'CFPS_fy4_est', 'CFPS_fy5_est',
    'NumAnalysts_fy1', 'NumAnalysts_fy2', 'NumAnalysts_fy3', 'NumAnalysts_fy4', 'NumAnalysts_fy5',
    'LTG_est',
    'shares_outstanding',
]
existing_keep = [c for c in keep_cols if c in analyst_df.columns]
out = analyst_df[existing_keep].copy().sort_values(['firm_id', 'date']).reset_index(drop=True)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print('rows=', len(out), '| companies=', out['firm_id'].nunique(), '| date range:', out['date'].min(), '->', out['date'].max())
